# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jenilrupareliya5150-bit/FlyRankAi-ml-Track/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [25]:
import pandas as pd
import numpy as np

In [26]:
!git clone https://github.com/jenilrupareliya5150-bit/FlyRankAi-ml-Track.git

Cloning into 'FlyRankAi-ml-Track'...
remote: Enumerating objects: 149, done.
remote: Counting objects: 100% (149/149), done.
remote: Compressing objects: 100% (105/105), done.
remote: Total 149 (delta 59), reused 90 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (149/149), 1.87 MiB | 13.03 MiB/s, done.
Resolving deltas: 100% (59/59), done.


In [27]:
%cd FlyRankAi-ml-Track

/content/FlyRankAi-ml-Track/FlyRankAi-ml-Track/FlyRankAi-ml-Track


In [28]:
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

In [29]:
a=df.columns.tolist()
np.array(a)

array(['content_id', 'client_id', 'search_volume', 'competition',
       'competition_level', 'cpc', 'content_type', 'main_intent',
       'word_count', 'char_count', 'provider_used', 'model_used',
       'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d',
       'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d',
       'scroll_events_90d', 'days_with_impressions', 'days_with_sessions',
       'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d',
       'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d',
       'content_age_days', 'age_tier', 'age_tier_order',
       'days_since_last_update', 'freshness_tier', 'word_count_tier',
       'char_count_tier', 'ctr', 'avg_position', 'engagement_rate',
       'scroll_rate', 'ai_traffic_pct', 'impression_tier',
       'position_tier', 'trend_direction', 'trend_pct'], dtype='<U22')

## 1. Unit of analysis + time window


The unit of analysis is **one content page (one webpage)**. Each row represents a single webpage identified by a unique `content_id`.

The dataset contains features aggregated over a **90-day historical window**, as shown by fields such as `impressions_90d`, `clicks_90d`, and `sessions_90d`. It also includes metrics from the previous and last 30-day periods, indicating that the dataset summarizes historical SEO performance before the prediction target is created.

In [30]:
print("Dataset Shape:", df.shape)

print("\nUnique Content Pages:", df["content_id"].nunique())
print("Total Rows:", len(df))

if df["content_id"].nunique() == len(df):
    print("\n✅ One row represents one unique webpage.")
else:
    print("\n⚠ Multiple rows exist for the same webpage.")

print("\nHistorical Window Columns:")

history_cols = [col for col in df.columns if "90d" in col or "30d" in col]

for col in history_cols:
    print("-", col)

Dataset Shape: (30000, 44)

Unique Content Pages: 30000
Total Rows: 30000

✅ One row represents one unique webpage.

Historical Window Columns:
- impressions_90d
- clicks_90d
- pageviews_90d
- sessions_90d
- users_90d
- engaged_sessions_90d
- ai_sessions_90d
- scroll_events_90d
- impressions_last_30d
- clicks_last_30d
- sessions_last_30d
- impressions_prev_30d
- clicks_prev_30d
- sessions_prev_30d


## 2. Fields: feature / label / context / excluded


The dataset fields are grouped into four categories:

- **Features:** SEO and content signals available before prediction, such as search volume, competition, CPC, content type, user intent, historical traffic metrics, engagement metrics, and freshness-related features.
- **Label:** `trend_direction`, which represents the future trend that the model will predict.
- **Context:** `content_id` and `client_id`, which identify webpages and clients but are not used for model training.
- **Excluded:** `trend_pct`, because it is used to derive the label (`trend_direction`). Including it would leak target information into the model and produce unrealistic performance.

In [31]:
feature_cols = [
    "search_volume",
    "competition",
    "competition_level",
    "cpc",
    "content_type",
    "main_intent",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "content_age_days",
    "days_since_last_update"
]

context_cols = ["content_id", "client_id"]
label_col = "trend_direction"
excluded_cols = ["trend_pct"]

print("Feature Columns:", len(feature_cols))
print(feature_cols)

print("\nLabel Column:")
print(label_col)

print("\nContext Columns:")
print(context_cols)

print("\nExcluded Columns:")
print(excluded_cols)

Feature Columns: 17
['search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'sessions_90d', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'content_age_days', 'days_since_last_update']

Label Column:
trend_direction

Context Columns:
['content_id', 'client_id']

Excluded Columns:
['trend_pct']


## 3. Verify it with queries (grain, counts, missing values, windows)



The following queries verify the main assumptions of the data contract.

- Confirm that each row represents one unique content page.
- Confirm the total number of rows and unique pages.
- Measure missing values in important feature columns.
- Check whether missing values follow a pattern across different content types.
- Verify that the dataset contains historical 90-day and 30-day window features.

In [32]:
# -------------------------------
# 1. Verify grain
# -------------------------------

print("Total Rows:", len(df))
print("Unique Content Pages:", df["content_id"].nunique())

# -------------------------------
# 2. Missing values
# -------------------------------

important_cols = [
    "search_volume",
    "competition",
    "cpc",
    "ctr",
    "avg_position"
]

print("\nMissing Values")
print(df[important_cols].isnull().sum())

# -------------------------------
# 3. Do missing values follow a pattern?
# -------------------------------

print("\nMissing Search Volume by Content Type")

missing_pattern = (
    df.groupby("content_type")["search_volume"]
      .apply(lambda x: x.isnull().sum())
)

print(missing_pattern)

# -------------------------------
# 4. Verify historical windows
# -------------------------------

window_cols = [col for col in df.columns if "90d" in col or "30d" in col]

print("\nHistorical Window Columns")

for col in window_cols:
    print("-", col)

Total Rows: 30000
Unique Content Pages: 30000

Missing Values
search_volume    2468
competition      2468
cpc              2468
ctr                 0
avg_position        0
dtype: int64

Missing Search Volume by Content Type
content_type
comparison article       0
feedly article        2096
keyword article        372
Name: search_volume, dtype: int64

Historical Window Columns
- impressions_90d
- clicks_90d
- pageviews_90d
- sessions_90d
- users_90d
- engaged_sessions_90d
- ai_sessions_90d
- scroll_events_90d
- impressions_last_30d
- clicks_last_30d
- sessions_last_30d
- impressions_prev_30d
- clicks_prev_30d
- sessions_prev_30d


## 4. Data limits

This dataset has several important limitations:

- It is a sample of the full warehouse and does not contain every webpage or client.
- The features are based on historical SEO performance, so they describe past behavior rather than guaranteeing future outcomes.
- Important factors such as backlinks, website speed, competitor actions, and Google algorithm updates are not included.
- The model can identify patterns and support refresh decisions, but it cannot prove that refreshing a page will directly improve its ranking or traffic.

In [34]:
print("Dataset Summary")
print("----------------")
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Unique Clients:", df["client_id"].nunique())
print("Unique Content Pages:", df["content_id"].nunique())

Dataset Summary
----------------
Rows: 30000
Columns: 44
Unique Clients: 32
Unique Content Pages: 30000


## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.